In [1]:
import json
import re
from pathlib import Path
from collections import Counter

In [2]:
DATA_DIR = Path("data")
STATS_DIR = Path(".")
ARTICLE_PATTERN = re.compile(r"\b(?:Article\s+\d+|Art\.\s+\d+)")

In [3]:
# get all metadata from last json file

most_recent_file = max(
    (f for f in STATS_DIR.iterdir() if f.suffix == ".json"),
    key=lambda f: f.stat().st_mtime
)

with open(most_recent_file, "r", encoding="utf-8") as f:
    data = json.load(f)

In [4]:
# filter to remove document not indexed

indexed_data = {
    url: entry
    for url, entry in data.items()
    if entry.get("is_indexed") is True
}

In [5]:
print(f"Nb documents in corpus: {len(indexed_data)}")

Nb documents in corpus: 652


In [6]:
cats_lang_combinations = Counter()

for _, entry in indexed_data.items():
    cats = tuple(sorted(entry.get("cats")))
    lang = entry.get("lang")
    key = (lang, cats)
    cats_lang_combinations[key] += 1

print(f"Stats on categories:")
for (lang, combo), count in cats_lang_combinations.most_common():
    print(f"  lang={lang} | cats={list(combo)}: {count}")

Stats on categories:
  lang=fr | cats=['summary']: 168
  lang=en | cats=['summary']: 168
  lang=fr | cats=['lex']: 153
  lang=en | cats=['lex']: 118
  lang=en | cats=['appendix']: 22
  lang=fr | cats=['appendix']: 21
  lang=fr | cats=['appendix', 'lex']: 1
  lang=en | cats=['appendix', 'lex']: 1


In [7]:
source_lang_combinations = Counter()

for _, entry in indexed_data.items():
    lang = entry.get("lang")
    source = entry.get("source")
    key = (source, lang)
    source_lang_combinations[key] += 1

print(f"Stats on sources:")
for (source, lang), count in source_lang_combinations.most_common():
    print(f"  source={source} | lang={lang}: {count}")

Stats on sources:
  source=polylex | lang=fr: 168
  source=polylex | lang=en: 168
  source=others | lang=fr: 146
  source=others | lang=en: 136
  source=fedlex | lang=fr: 29
  source=fedlex | lang=en: 5


In [8]:
format_lang_combinations = Counter()

for _, entry in indexed_data.items():
    content_format = entry.get("content_format")
    lang = entry.get("lang")
    key = (lang, content_format)
    format_lang_combinations[key] += 1

print(f"Stats on content formats:")
for (lang, content_format), count in format_lang_combinations.most_common():
    print(f"  lang={lang} | format={content_format}: {count}")

Stats on content formats:
  lang=fr | format=pdf: 174
  lang=fr | format=txt: 168
  lang=en | format=txt: 168
  lang=en | format=pdf: 140
  lang=fr | format=docx: 1
  lang=en | format=docx: 1


In [9]:
unique_refs = {
    (
        ref.get("lex_id"),
        ref.get("lex_type"),
        ref.get("lex_number"),
        ref.get("lex_lang")
    )
    for entry in indexed_data.values()
    for ref in entry.get("refs", [])
}

print(f"Nb unique refs: {len(unique_refs)}")

Nb unique refs: 338


In [10]:
count = sum(
    1
    for entry in indexed_data.values()
    if len(entry.get("refs", [])) > 1
)

print(f"Nb duplicated docs: {count}")

Nb duplicated docs: 35


In [12]:
results = {
    content: entry
    for content, entry in indexed_data.items()
    if "summary" in entry.get("cats")
    and len(entry.get("refs")) > 1
}

print(f"Nb summaries duplicated: {len(results)}")
for content in results:
    print(f"Content:\n  {content}")

Nb summaries duplicated: 2
Content:
  Tax ruling EPFL (Valais)
Exonération de l'Ecole polytechnique fédérale de Lausanne (EPFL) en matière d'impôts sur les successions et sur les donations.
Content:
  EPFL Mission and Strategic Outlook
EPFL Mission and Strategic Outlook 2025-2028.
